In [1]:
# ============================================================================
# 05a — BUILD THE BLINDED AUDIT WORKSHEET
# ----------------------------------------------------------------------------
# Samples 20 habit-consistent and 20 habit-contradicting interactions, shuffles
# them so the annotator cannot tell which is which, and prints the supporting
# PubMed links for each.
#
# Every protein is listed with its synonyms, because the cited papers often use
# names from the 1980s-90s (TCF4 appears as "E2-2", KDR as "VEGFR2"). Without
# them the annotator cannot find the proteins in the abstracts at all.
#
# Output: docs/label_audit_blinded.md  (worksheet)
#         docs/label_audit_key.csv     (answer key — do not open until done)
# ============================================================================
from pathlib import Path
if (Path.cwd().parent/"docs"/"label_audit_blinded.md").exists():
    print("worksheet already filled in docs/label_audit_blinded.md; not rebuilding it")
else:
    import json, urllib.request, urllib.parse, time
    from pathlib import Path
    import pandas as pd, numpy as np

    ROOT = Path.cwd().parent
    RAW, PROC, DOCS = ROOT/"data"/"raw", ROOT/"data"/"processed", ROOT/"docs"
    DOCS.mkdir(exist_ok=True)

    B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)

    # ---- references behind each edge ----------------------------------------
    ref_map = {}
    for n in ["causal_ppi","collectri","kinaseextra"]:
        d = pd.read_parquet(RAW/f"{n}.parquet")
        if "references" not in d.columns: continue
        for s,t,r in zip(d.source_genesymbol, d.target_genesymbol, d["references"]):
            if isinstance(r,str) and r.strip():
                ref_map.setdefault((s,t), set()).update(
                    x.split(":")[-1] for x in r.split(";")
                    if x.strip().split(":")[-1].isdigit())
    B["pmids"] = [sorted(ref_map.get((r.source,r.target), set()))[:3] for _,r in B.iterrows()]

    # ---- every alias, from UniProt ------------------------------------------
    cache = PROC/"gene_aliases.json"
    if cache.exists():
        ALIAS = json.loads(cache.read_text()); print(f"aliases cached ({len(ALIAS)})")
    else:
        acc = {}
        for _, r in B.iterrows():
            if isinstance(r.source_uniprot,str): acc.setdefault(r.source, r.source_uniprot)
            if isinstance(r.target_uniprot,str): acc.setdefault(r.target, r.target_uniprot)
        a2g = {v:k for k,v in acc.items()}
        accs, ALIAS = sorted(acc.values()), {}
        print(f"fetching synonyms for {len(accs)} proteins...")
        for i in range(0, len(accs), 50):
            q = "(" + " OR ".join(f"accession:{a}" for a in accs[i:i+50]) + ")"
            try:
                u = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
                     f"&fields=accession,gene_names,protein_name&query={urllib.parse.quote(q)}")
                txt = urllib.request.urlopen(u, timeout=120).read().decode()
                for ln in txt.splitlines()[1:]:
                    p = ln.split("\t")
                    if len(p) >= 2 and p[0] in a2g:
                        g = a2g[p[0]]
                        syn = [x for x in p[1].split() if x.upper() != g.upper()]
                        ALIAS[g] = dict(aliases=syn[:5],
                                        name=(p[2][:60] if len(p) > 2 else ""))
            except Exception as e:
                print(f"  batch {i} failed: {type(e).__name__}")
            time.sleep(0.2)
        cache.write_text(json.dumps(ALIAS)); print(f"  got {len(ALIAS)}")

    def names(g):
        a = ALIAS.get(g, {}).get("aliases", [])
        return f"**{g}**" + (f" *(also: {', '.join(a)})*" if a else "")

    # ---- sample, blinded -----------------------------------------------------
    rng = np.random.RandomState(42)
    S = pd.concat([
        B[(B.condition=="habit_misleads") & (B.pmids.map(len)>0)].sample(20, random_state=1),
        B[(B.condition=="habit_agrees")   & (B.pmids.map(len)>0)].sample(20, random_state=2)])
    S = S.iloc[rng.permutation(len(S))].reset_index(drop=True)
    S["audit_id"] = [f"A{i+1:02d}" for i in range(len(S))]

    L = ["# Label audit — blinded", "",
    "## How to judge each one", "",
    "Open the papers. Ask: **does this paper show that the first protein",
    "decreases or increases the activity or level of the second?**", "",
    "| Mark | When |",
    "|------|------|",
    "| **Y** | the paper supports the stated direction |",
    "| **N** | the paper shows the OPPOSITE direction |",
    "| **?** | ambiguous, context-dependent, or the paper never addresses this pair |",
    "| **X** | the paper is not about these two proteins at all |", "",
    "### Reading tips", "",
    "- Papers often use OLD names. The aliases are listed for you — search the",
    "  abstract for any of them.",
    "- 'represses / suppresses / blocks / downregulates' = **inhibits**",
    "- 'induces / activates / upregulates / promotes / required for' = **stimulates**",
    "- Promoter or reporter assays count: repressing a promoter is inhibition.",
    "- If the abstract is not enough, the full text usually is — but if you cannot",
    "  tell in ~2 minutes, mark **?** and move on. '?' is a real answer.", "",
    "**Do not look at the answer key until you have finished all 40.**", "",
    "---", ""]

    for _, r in S.iterrows():
        L += [f"### {r.audit_id}", "",
              f"**Claim:** {r.source} → {r.target} — OmniPath says "
              f"**{r.answer.upper()}**", "",
              f"- upstream: {names(r.source)}",
              f"  <br>*{ALIAS.get(r.source,{}).get('name','')}*",
              f"- downstream: {names(r.target)}",
              f"  <br>*{ALIAS.get(r.target,{}).get('name','')}*", "",
              "**Papers:** " + "  ".join(
                  f"[{p}](https://pubmed.ncbi.nlm.nih.gov/{p}/)" for p in r.pmids), "",
              "**Verdict:** `   `  (Y / N / ? / X)", "",
              "**Note:** ", "", "---", ""]

    (DOCS/"label_audit_blinded.md").write_text("\n".join(L), encoding="utf-8")
    S[["audit_id","source","target","answer","condition","protein_usually",
       "n_refs","component"]].to_csv(DOCS/"label_audit_key.csv", index=False)

    print(f"\nworksheet: {DOCS/'label_audit_blinded.md'}  ({len(S)} items)")
    print(f"key      : {DOCS/'label_audit_key.csv'}  — do not open yet")
    print("\nExample of the new format:")
    r = S.iloc[0]
    print(f"\n  Claim: {r.source} -> {r.target}, OmniPath says {r.answer.upper()}")
    print(f"  upstream  : {names(r.source)}")
    print(f"  downstream: {names(r.target)}")

worksheet already filled in docs/label_audit_blinded.md; not rebuilding it


In [2]:
# ============================================================================
# 05b — SCORE THE AUDIT
# ----------------------------------------------------------------------------
# Verdicts are read from the annotator's worksheet (the single source of truth):
# Y supported, N opposite, ? undetermined (blank counts as ?), X paper not about
# these proteins. Items where the cited papers disagree were scored on the
# majority and are flagged. The worst-case sensitivity assumes every mislabelled
# item flips the model's score; it is reported one-sided (exception labels only)
# and two-sided (both conditions).
# ============================================================================
import re, shutil
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import fisher_exact, beta

ROOT = Path.cwd().parent
PROC, RES = ROOT/"data"/"processed", ROOT/"results"
ONESIDED_CUTOFF = 0.85
A = pd.concat([pd.read_json(p, lines=True) for p in sorted(RES.glob("*.jsonl"))], ignore_index=True)
A = A[A.http==200].copy()
A["abstained"] = A.pred=="not_established"
A["committed"] = A.pred.isin(["stimulates","inhibits"])
A["correct"]   = A.pred.notna() & (A.pred==A.truth)
MAIN = A[(A.arm=="main") & ~A.is_decoy]; DEC = A[(A.arm=="main") & A.is_decoy]
SW, WEB, U = A[A.arm=="swap"], A[A.arm=="web"], A[A.arm=="usually"]
MODELS_ = sorted(MAIN.model.unique()); PROTS = sorted(MAIN.source.unique())
skew = {m: MAIN[(MAIN.model==m) & MAIN.committed].pred.value_counts(normalize=True).max() for m in MODELS_}
ONESIDED = [m for m in MODELS_ if skew[m] > ONESIDED_CUTOFF]
INFORM   = [m for m in MODELS_ if m not in ONESIDED]
def acc(d, c): return d[d.condition==c].correct.mean()
def gap(d, m=None):
    if m is not None: d = d[d.model==m]
    return (acc(d,"habit_agrees") - acc(d,"habit_misleads"))*100
DOCS, SUPP = ROOT/"docs", ROOT/"supplementary"/"tables"; SUPP.mkdir(parents=True, exist_ok=True)
CONFLICT = {"A14", "A15", "A23"}      # cited papers disagree; scored on the majority

md = (DOCS/"label_audit_blinded.md").read_text(encoding="utf-8")
parts = re.split(r"^### (A\d\d)\s*$", md, flags=re.M)
ws = {}
for i in range(1, len(parts), 2):
    body = parts[i+1]
    v = re.search(r"\*\*Verdict:\*\*\s*`([^`]*)`", body)
    n = re.search(r"\*\*Note:\*\*(.*?)(?:\n---|\Z)", body, flags=re.S)
    raw_v = (v.group(1).strip() if v else "")
    ws[parts[i]] = dict(verdict=(raw_v[:1].upper() if raw_v else "?"), note=(n.group(1).strip() if n else ""))
K = pd.read_csv(DOCS/"label_audit_key.csv")
K["verdict"] = K.audit_id.map(lambda a: ws[a]["verdict"])
K["papers_conflict"] = K.audit_id.isin(CONFLICT)
K["supporting_sentence"] = K.audit_id.map(lambda a: ws[a]["note"])
print(f"verdicts: {K.verdict.value_counts().to_dict()}  (Y supported, N opposite, ? undetermined, X not about the pair)")
V = K[K.verdict.isin(["Y","N"])]
print(f"verdict reached for {len(V)}/40; supported {int((V.verdict=='Y').sum())}/{len(V)} "
      f"({(V.verdict=='Y').mean():.0%}); undetermined or not about the pair: {40-len(V)} ({(40-len(V))/40:.0%})")
err = {}
for c in ["habit_agrees","habit_misleads"]:
    s = V[V.condition==c]; k = int((s.verdict=="N").sum()); n = len(s); err[c] = k/n
    lo = beta.ppf(.025, k, n-k+1) if k else 0.0; hi = beta.ppf(.975, k+1, n-k)
    print(f"  {c:<15} supported {n-k}/{n}; contradicted {k}/{n} = {k/n:.1%} (exact 95% CI {lo:.1%}-{hi:.1%})")
tab = pd.crosstab(V.condition, V.verdict).reindex(columns=["Y","N"], fill_value=0)
print(f"  Fisher exact p = {fisher_exact(tab.values)[1]:.2f}")
print(f"  conflicting-evidence items scored on the majority: {sorted(CONFLICT)}")
print(f"  Y/N verdicts without a recorded supporting sentence: "
      f"{list(V[V.supporting_sentence.str.len()==0].audit_id)}")

print("\nWORST-CASE SENSITIVITY (informative models)")
g0 = np.array([gap(MAIN, m) for m in INFORM])
for lab_, e in [("exception labels only", err["habit_misleads"]),
                ("both conditions", err["habit_misleads"] + err["habit_agrees"])]:
    g = g0 - e*100
    print(f"  {lab_:<22} gap shrinks by {e*100:.1f} pp -> median {np.median(g):.1f}, "
          f"positive {int((g>0).sum())}/{len(INFORM)}; not positive: {[m for m, x in zip(INFORM, g) if x <= 0]}")

K.to_csv(DOCS/"label_audit_unblinded.csv", index=False)
K.to_csv(SUPP/"TableS2_label_audit.csv", index=False)
shutil.copy(DOCS/"label_audit_blinded.md", SUPP/"TableS2_audit_worksheet.md")
print(f"\nwritten: docs/label_audit_unblinded.csv, supplementary/tables/TableS2_label_audit.csv (+ worksheet)")

verdicts: {'Y': 20, '?': 16, 'N': 3, 'X': 1}  (Y supported, N opposite, ? undetermined, X not about the pair)
verdict reached for 23/40; supported 20/23 (87%); undetermined or not about the pair: 17 (42%)
  habit_agrees    supported 10/11; contradicted 1/11 = 9.1% (exact 95% CI 0.2%-41.3%)
  habit_misleads  supported 10/12; contradicted 2/12 = 16.7% (exact 95% CI 2.1%-48.4%)
  Fisher exact p = 1.00
  conflicting-evidence items scored on the majority: ['A14', 'A15', 'A23']
  Y/N verdicts without a recorded supporting sentence: ['A01']

WORST-CASE SENSITIVITY (informative models)
  exception labels only  gap shrinks by 16.7 pp -> median 13.9, positive 14/14; not positive: []
  both conditions        gap shrinks by 25.8 pp -> median 4.8, positive 12/14; not positive: ['gpt-oss-120b', 'med42-70b']

written: docs/label_audit_unblinded.csv, supplementary/tables/TableS2_label_audit.csv (+ worksheet)
